In [0]:
from pyspark.sql.functions import when,col,hour,date_format,from_json,schema_of_json,size,current_timestamp

In [0]:
from pyspark import pipelines as dp

In [0]:
@dp.table(
    name = "restaurant.silver.fact_orders",
    table_properties= {"quality":"silver"}
)
@dp.expect_or_fail("order_id_not_null","order_id IS NOT NULL")
def silver_fact_orders():
    df = spark.readStream.table("restaurant.bronze.historical_orders") \
            .withColumn("order_timestamp",col("timestamp").cast("timestamp")) \
            .withColumn("order_date",col("timestamp").cast("date")) \
            .withColumn("order_hour",hour(col("timestamp"))) \
            .withColumn("day_of_week",date_format(col("timestamp"),"EEEE")) \
            .withColumn("is_weekend",when(col("day_of_week").isin("Saturday","Sunday") ,True).otherwise(False)) \
            .withColumn("items",from_json(col("items"),schema_of_json('[{"item_id": "ITEM-302", "name": "Chicken Tikka Masala", "category": "Main Course", "quantity": 2, "unit_price": 52.43, "subtotal": 104.86}]'))) \
            .withColumn("item_count",size(col("items"))) \
            .withColumn("_ingestion_timestamp",current_timestamp()) \
            .select("order_id","order_timestamp","order_date","order_hour","day_of_week","is_weekend","restaurant_id","customer_id","order_type","item_count","total_amount","payment_method","order_status","_ingestion_timestamp")
    return df